# Managing data for Songs

In [4]:
import pandas as pd
import os
import librosa

Get the directory for constructing the songs path (in alternative you can compute arrays)

In [9]:
work_dir = os.getcwd().removesuffix("/data_management")
work_dir

In [12]:
DEAM_dataset = work_dir + "/data/DEAM"
PMEMO_dataset = work_dir + "/data/PMEmo2019"

## PMEMo

In [13]:
pmemo_annotations = PMEMO_dataset + "/annotations/static_annotations.csv"
pmemo_songs = PMEMO_dataset + "/chorus"
pmemo_metadata= PMEMO_dataset + "/metadata.csv"

PMEMO_ann= pd.read_csv(pmemo_annotations, sep =",")
PMEMO_metadata = pd.read_csv(pmemo_metadata, sep =",")


In [14]:
PMEMO_ann

,musicId,Arousal(mean),Valence(mean)
0,1,0.4000,0.5750
1,4,0.2625,0.2875
2,5,0.1500,0.2000
3,6,0.5125,0.3500
4,7,0.7000,0.7250
...,...,...,...
762,993,0.8625,0.7625
763,996,0.8750,0.5625
764,997,0.7125,0.6625
765,999,0.8750,0.7750


Retrieving MP3 files (alternative)

In [ ]:
mp3_df = pd.DataFrame([], columns=["musicId", "mp3_file"])

for file in os.listdir(pmemo_songs):
    data = os.path.join(pmemo_songs, file)
    audio, sr = librosa.load(data, sr = 44100)
    mp3_df = pd.concat([mp3_df, pd.DataFrame([{"musicId": file.replace(".mp3",""), "mp3_file" :audio}])])

mp3_df

Retrieving paths

In [ ]:
mp3_path = pd.DataFrame([], columns=["musicId", "mp3_path"])

for file in os.listdir(pmemo_songs):
    data = os.path.join(pmemo_songs, file)
    mp3_path = pd.concat([mp3_path, pd.DataFrame([{"musicId": file.replace(".mp3",""), "mp3_path": data}])])

mp3_path

Merging features and files (paths)

In [16]:
PMEMO_ann["musicId"] = PMEMO_ann["musicId"].astype(str)
PMEMO_metadata["musicId"] = PMEMO_metadata["musicId"].astype(str)
mp3_path["musicId"] = mp3_path["musicId"].astype(str)

pmemo = PMEMO_ann.join(PMEMO_metadata.set_index("musicId"), "musicId")\
                 .join(mp3_path.set_index("musicId"), "musicId")\
                 .drop(columns=["fileName","duration", "chorus_start_time", "chorus_end_time"])\
                 .rename(columns={"Valence(mean)": "Valence", "Arousal(mean)":"Arousal"})



In [17]:
pmemo

,musicId,Arousal,Valence,title,artist,album,mp3_path
0,1,0.4000,0.5750,Good Drank,2 Chainz,"Def Jam Presents: Direct Deposit, Vol. 2",/Users/lucabellani/Documents/UNI/Tesi/Recommer...
1,4,0.2625,0.2875,X Bitch (feat. Future),21 Savage,Savage Mode,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
2,5,0.1500,0.2000,No Heart,21 Savage,Savage Mode,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
3,6,0.5125,0.3500,Red Opps,21 Savage,Red Opps,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
4,7,0.7000,0.7250,Girls Talk Boys,5 Seconds Of Summer,Ghostbusters (Original Motion Picture Soundtrack),/Users/lucabellani/Documents/UNI/Tesi/Recommer...
...,...,...,...,...,...,...,...
762,993,0.8625,0.7625,Stay,Zedd,Stay,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
763,996,0.8750,0.5625,Trouble,offaiah,Trouble,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
764,997,0.7125,0.6625,A Supplementary Story : You Never Walk Alone,방탄소년단,YOU NEVER WALK ALONE,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
765,999,0.8750,0.7750,Outro : Wings,방탄소년단,YOU NEVER WALK ALONE,/Users/lucabellani/Documents/UNI/Tesi/Recommer...


In [14]:
pmemo.to_pickle("./data/PMEmo_useful")

## DEAM

In [18]:
deam_metadata_2013 = pd.read_csv(DEAM_dataset+"/DEAM_metadata/metadata_2013.csv", sep =",")
deam_metadata_2014 = pd.read_csv(DEAM_dataset+"/DEAM_metadata/metadata_2014.csv", sep =",")                    
deam_metadata_2015 = pd.read_csv(DEAM_dataset+"/DEAM_metadata/metadata_2015.csv", sep =",")

Drop uninterested columns

In [19]:
deam_metadata_2013 = deam_metadata_2013.drop(columns=["start of the segment (min.sec)", "end of the segment (min.sec)", "Genre", "file_name"])\
                                       .rename(columns = {"song_id": "musicId", "Artist" : "artist", "Song title" : "title"})

deam_metadata_2013["artist"] = deam_metadata_2013["artist"].map(lambda x:x.replace("\t",""))
deam_metadata_2013["title"] = deam_metadata_2013["title"].map(lambda x:x.replace("\t",""))

deam_metadata_2013

,musicId,artist,title
0,2,The New Mystikal Troubadours,Tonight A Lonely Century
1,3,Kevin MacLeod,DD Groove
2,4,Kevin MacLeod,Slow Burn
3,5,My Bubba & Mi,Nothing Much
4,7,Kevin MacLeod,Hustle
...,...,...,...
739,995,Benjamin Bret,Honte De Toi
740,996,Jahzzar,Fireworks
741,997,Halloween,Monster On Campus
742,999,Jahzzar,So Easy


In [20]:
deam_metadata_2014 = deam_metadata_2014.drop(columns = ["segment start", "segment end", "last.fm labels"])
deam_metadata_2014.columns

Index(['Id', 'Artist', 'Album', 'Track', 'Genre'], dtype='object')

In [21]:
deam_metadata_2015 = deam_metadata_2015.drop(columns=["Filename", "album", "genre", "Unnamed: 6"])\
                                       .rename(columns = {"id": "musicId", "Artist" : "artist", "Song title" : "title"})
deam_metadata_2015

,musicId,title,artist
0,2001,Old Tree,Creepoid
1,2002,Spring Day 1,Amar Lal
2,2003,Excessive Resistance to Change,Joel Helander
3,2004,Flying,Aimee Norwich
4,2005,Waterduct,Ava Luna
5,2006,Air Traffic,Clara Berry and Wooldog
6,2007,Heavy Love,Dreamers of the Ghetto
7,2008,Disturbing Wildlife,Invisible Familiars
8,2009,Definition,Joel Helander
9,2010,Yatora,Karim Douaidy


Merging DEAM 2013 and DEAM 2015 (2014 is not manageble). 

In [22]:
#without metadata 2014
deam_metadata = pd.concat([deam_metadata_2013, deam_metadata_2015])
deam_metadata

,musicId,artist,title
0,2,The New Mystikal Troubadours,Tonight A Lonely Century
1,3,Kevin MacLeod,DD Groove
2,4,Kevin MacLeod,Slow Burn
3,5,My Bubba & Mi,Nothing Much
4,7,Kevin MacLeod,Hustle
...,...,...,...
53,2054,Tom La Meche,Interlude
54,2055,Goo Goo Cluster,Vide grenier
55,2056,Ruediger Kramer,happy child singing
56,2057,La Verue,Au Feu


Reading the annotations and normalizing valence and arousal

In [23]:
deam_ann_2000 = pd.read_csv(DEAM_dataset+"/annotations/annotations averaged per song/song_level/static_annotations_averaged_songs_1_2000.csv", sep =",")
deam_ann_2058 = pd.read_csv(DEAM_dataset+"/annotations/annotations averaged per song/song_level/static_annotations_averaged_songs_2000_2058.csv", sep =",")

useful_columns = ["song_id", " valence_mean", " arousal_mean"]
deam_ann_2058 = deam_ann_2058[useful_columns].rename(columns = {"song_id":"musicId"," valence_mean": "Valence", " arousal_mean":"Arousal"})
deam_ann_2000 = deam_ann_2000[useful_columns].rename(columns = {"song_id":"musicId", " valence_mean": "Valence", " arousal_mean":"Arousal"})

print(deam_ann_2000.columns)
def normalize(col, min, max):
    return (col - min)/(max - min)

deam_ann_2000["Valence"] = normalize(deam_ann_2000["Valence"], deam_ann_2000["Valence"].min(), deam_ann_2000["Valence"].max())
deam_ann_2000["Arousal"] = normalize(deam_ann_2000["Arousal"], deam_ann_2000["Arousal"].min(), deam_ann_2000["Arousal"].max())

deam_ann_2058["Valence"] = normalize(deam_ann_2058["Valence"], deam_ann_2058["Valence"].min(), deam_ann_2058["Valence"].max())
deam_ann_2058["Arousal"] = normalize(deam_ann_2058["Arousal"], deam_ann_2058["Arousal"].min(), deam_ann_2058["Arousal"].max())

print(deam_ann_2000)
print(deam_ann_2058)

Index(['musicId', 'Valence', 'Arousal'], dtype='object')
      musicId   Valence   Arousal
0           2  0.220588  0.215385
1           3  0.279412  0.261538
2           4  0.602941  0.600000
3           5  0.411765  0.569231
4           7  0.617647  0.738462
...       ...       ...       ...
1739     1996  0.338235  0.661538
1740     1997  0.544118  0.353846
1741     1998  0.705882  0.707692
1742     1999  0.441176  0.584615
1743     2000  0.617647  0.676923

[1744 rows x 3 columns]
    musicId   Valence  Arousal
0      2001  0.047619   0.9000
1      2002  0.809524   0.5500
2      2003  0.571429   0.4000
3      2004  0.476190   0.4500
4      2005  0.190476   0.5500
5      2006  0.476190   0.0500
6      2007  0.095238   0.3500
7      2008  0.619048   0.1500
8      2009  0.523810   0.3000
9      2010  0.714286   0.4500
10     2011  0.285714   0.5000
11     2012  0.040476   0.5425
12     2013  0.571429   0.6500
13     2014  0.000000   0.9000
14     2015  0.857143   0.6000
15     2016  1

In [24]:
deam_ann = pd.concat([deam_ann_2000[deam_ann_2000["musicId"] <= 1000], deam_ann_2058])

deam_ann

,musicId,Valence,Arousal
0,2,0.220588,0.215385
1,3,0.279412,0.261538
2,4,0.602941,0.600000
3,5,0.411765,0.569231
4,7,0.617647,0.738462
...,...,...,...
53,2054,0.571429,0.150000
54,2055,0.476190,0.550000
55,2056,0.476190,0.400000
56,2057,0.040476,0.957500


Retrieving filepaths

In [26]:
deam_path = pd.DataFrame([], columns=["musicId", "mp3_path"])
deam_audio = DEAM_dataset+"/MEMD_audio" 

for file in os.listdir(deam_audio):
    deam_path = pd.concat([deam_path, pd.DataFrame([{"musicId": file.replace(".mp3",""), \
                                                    "mp3_path" :os.path.join(deam_audio, file)}])])

Numpy arrays alternative

In [ ]:
deam_audio = DEAM_dataset+"/MEMD_audio" 

deam_mp3 = pd.DataFrame([], columns=["musicId", "mp3_file"])

for file in os.listdir(deam_audio):
    audio, sr = librosa.load(os.path.join(deam_audio, file), sr=44100)
    deam_mp3 = pd.concat([deam_mp3, pd.DataFrame([{"musicId": file.replace(".mp3",""), "mp3_file" :audio}])])


Merging metadata and files and setting new indexes (for concating datasets in end)

In [27]:
deam_ann["musicId"] = deam_ann["musicId"].astype("str")
deam_path["musicId"] = deam_path["musicId"].astype("str")
deam_metadata["musicId"] = deam_metadata["musicId"].astype("str")

deam = deam_ann.join(deam_path.set_index("musicId"),on = "musicId")\
               .join(deam_metadata.set_index("musicId"), on = "musicId")

deam["musicId"] = deam["musicId"].astype("int").map(lambda x: x+1000)

deam

,musicId,Valence,Arousal,mp3_path,artist,title
0,1002,0.220588,0.215385,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,The New Mystikal Troubadours,Tonight A Lonely Century
1,1003,0.279412,0.261538,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,Kevin MacLeod,DD Groove
2,1004,0.602941,0.600000,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,Kevin MacLeod,Slow Burn
3,1005,0.411765,0.569231,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,My Bubba & Mi,Nothing Much
4,1007,0.617647,0.738462,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,Kevin MacLeod,Hustle
...,...,...,...,...,...,...
53,3054,0.571429,0.150000,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,Tom La Meche,Interlude
54,3055,0.476190,0.550000,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,Goo Goo Cluster,Vide grenier
55,3056,0.476190,0.400000,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,Ruediger Kramer,happy child singing
56,3057,0.040476,0.957500,/Users/lucabellani/Documents/UNI/Tesi/Recommer...,La Verue,Au Feu


### Concatenation of PMEMo and DEAM

Concatenation 

In [28]:
songs = pd.concat([pmemo, deam]).drop(columns="album")
songs = songs.drop_duplicates(subset=['title', 'artist'], keep='last')
songs

,musicId,Arousal,Valence,title,artist,mp3_path
0,1,0.4000,0.575000,Good Drank,2 Chainz,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
1,4,0.2625,0.287500,X Bitch (feat. Future),21 Savage,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
2,5,0.1500,0.200000,No Heart,21 Savage,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
3,6,0.5125,0.350000,Red Opps,21 Savage,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
4,7,0.7000,0.725000,Girls Talk Boys,5 Seconds Of Summer,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
...,...,...,...,...,...,...
53,3054,0.1500,0.571429,Interlude,Tom La Meche,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
54,3055,0.5500,0.476190,Vide grenier,Goo Goo Cluster,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
55,3056,0.4000,0.476190,happy child singing,Ruediger Kramer,/Users/lucabellani/Documents/UNI/Tesi/Recommer...
56,3057,0.9575,0.040476,Au Feu,La Verue,/Users/lucabellani/Documents/UNI/Tesi/Recommer...


In [27]:
songs.to_pickle("../data/Songs_path")

In [ ]:
songs.to_pickle("../data/Songs")